In [ ]:
!pip install matplotlib torch torchvision tqdm numpy ipykernel torch-fidelity pandas

import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torch_fidelity
import pandas as pd

import os
import time
import pickle
import zipfile
import numpy as np
import urllib.request
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import random
import torchvision.models as models


seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ------------------------------------------------------------
# 0) Hyperparameters / config
# ------------------------------------------------------------
# Define some params
DATASET_PATH =  '/common/home/users/a/annamalaik.2022/cs-424-group-project-friday/task_2_data/'
DEVICE = 'cuda'  # 'cpu' or 'cuda'
OUTPUT_PATH = '/common/home/users/a/annamalaik.2022/cs-424-group-project-friday/work_dirs/Cycle_GAN_02'
CHECKPOINT_SAVE_EVERY = 2
LATENT_DIMS = 100
LR = 0.0002
WORKERS = 4
NETWORK_FEATURES = 128
EPOCH = 200
BATCH_SIZE = 8


# ------------------------------------------------------------
# 1) Define a custom Dataset that returns (cat_img, dog_img)
# ------------------------------------------------------------
# CycleGAN uses *unpaired* image-to-image translation:
# - We do NOT need aligned (apple_i, orange_i) pairs.
# - Instead, during training we sample an apple image from domain A
#   and an orange image from domain B independently.
#
# This dataset class does exactly that by indexing with modulo:
# - apple index cycles through trainA images
# - orange index cycles through trainB images
# and the length is max(len(A), len(B)) so an epoch covers the larger domain.
class Anime2RealDataset(torch.utils.data.Dataset):
    def __init__(self, root_dir, transform=None, train_or_test='train'):
        self.dir_A = os.path.join(root_dir, 'anime', train_or_test)
        self.dir_B = os.path.join(root_dir, 'real', train_or_test)

        self.transform = transform

        self.images_A = sorted([f for f in os.listdir(self.dir_A) if not f.startswith('.')])
        self.images_B = sorted([f for f in os.listdir(self.dir_B) if not f.startswith('.')])

        self.len_A = len(self.images_A)
        self.len_B = len(self.images_B)

        self.length_dataset = max(self.len_A, self.len_B)

    def __len__(self):
        return self.length_dataset

    def __getitem__(self, index):
        img_A = self.images_A[index % self.len_A]
        img_B = self.images_B[index % self.len_B]

        path_A = os.path.join(self.dir_A, img_A)
        path_B = os.path.join(self.dir_B, img_B)

        image_A = Image.open(path_A).convert("RGB")
        image_B = Image.open(path_B).convert("RGB")

        if self.transform:
            image_A = self.transform(image_A)
            image_B = self.transform(image_B)

        return image_A, image_B

transform = transforms.Compose([
    transforms.Resize(286),
    transforms.CenterCrop(256),   # more stable for faces
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)
])


dataset = Anime2RealDataset(DATASET_PATH, transform=transform, train_or_test='train')

dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=WORKERS,
    pin_memory=True,
    drop_last=True
)
print("hello task 2")

In [ ]:
# lambda_gan   = 5
# lambda_cycle = 10
# lambda_feat  = 0.02
# lambda_perc  = 0
# lambda_id    = 0

lambda_gan = 2
lambda_cycle = 10          # slightly stronger structure
lambda_feat = 0.1        # stabilize textures a bit more
lambda_id = 5          # identity loss is important for faces to preserve color and structure

In [ ]:
# -----------------------------
# MODEL BLOCKS
# -----------------------------
from torch.nn.utils import spectral_norm

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(channels, channels, 3),
            nn.InstanceNorm2d(channels)
        )

    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.ReflectionPad2d(3),
            nn.Conv2d(3, 64, 7),
            nn.InstanceNorm2d(64),
            nn.ReLU(True),

            nn.Conv2d(64, 128, 3, 2, 1),
            nn.InstanceNorm2d(128),
            nn.ReLU(True),

            nn.Conv2d(128, 256, 3, 2, 1),
            nn.InstanceNorm2d(256),
            nn.ReLU(True),

            *[ResidualBlock(256) for _ in range(9)],

            nn.ConvTranspose2d(256, 128, 3, 2, 1, output_padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 64, 3, 2, 1, output_padding=1),
            nn.InstanceNorm2d(64),
            nn.ReLU(True),

            nn.ReflectionPad2d(3),
            nn.Conv2d(64, 3, 7),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.layers = nn.ModuleList([
            nn.Sequential(
                spectral_norm(nn.Conv2d(3, 64, 4, 2, 1)),
                nn.LeakyReLU(0.2)
            ),
            nn.Sequential(
                spectral_norm(nn.Conv2d(64, 128, 4, 2, 1)),
                nn.InstanceNorm2d(128),
                nn.LeakyReLU(0.2)
            ),
            nn.Sequential(
                spectral_norm(nn.Conv2d(128, 256, 4, 2, 1)),
                nn.InstanceNorm2d(256),
                nn.LeakyReLU(0.2)
            ),
            nn.Sequential(
                spectral_norm(nn.Conv2d(256, 512, 4, 1, 1)),
                nn.InstanceNorm2d(512),
                nn.LeakyReLU(0.2)
            ),
            # Larger final kernal(5) compared to 
            # 4 in vanilla cycleGAN paper
            # spectral_norm(nn.Conv2d(512, 1, kernel=4, stride=1, padding=1))
            # Each output now “sees” a larger region of the input image
            # Increased the kernel size (4 → 5)
            # This increases the receptive field of each output pixel.
            # Each discriminator output now depends on a larger region of the input image
            # Task (anime ↔ real faces) requires more global context than typical texture translation tasks.
            spectral_norm(nn.Conv2d(512, 1, 5, 1, 2))
        ])

    def forward(self, x, return_features=False):
        feats = []
        for layer in self.layers[:-1]:
            x = layer(x)
            feats.append(x)
        out = self.layers[-1](x)

        if return_features:
            return out, feats
        return out

class MultiScaleDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.D1 = Discriminator()
        self.D2 = Discriminator()
        self.downsample = nn.AvgPool2d(3, stride=2, padding=1)

    def forward(self, x, return_features=False):
        x_down = self.downsample(x)

        if return_features:
            out1, feat1 = self.D1(x, return_features=True)
            out2, feat2 = self.D2(x_down, return_features=True)
            return [out1, out2], [feat1, feat2]
        else:
            return [self.D1(x), self.D2(x_down)]

In [ ]:
class ReplayBuffer:
    def __init__(self, max_size=50):
        self.data = []
        self.max_size = max_size

    def push_and_pop(self, batch):
        result = []
        for img in batch:
            img = img.unsqueeze(0)
            if len(self.data) < self.max_size:
                self.data.append(img)
                result.append(img)
            else:
                if random.random() > 0.5:
                    idx = random.randint(0, self.max_size - 1)
                    result.append(self.data[idx])
                    self.data[idx] = img
                else:
                    result.append(img)
        return torch.cat(result)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

anime_G = Generator().to(DEVICE)   # cat → dog
real_G = Generator().to(DEVICE)   # dog → cat

anime_D = MultiScaleDiscriminator().to(DEVICE)
real_D  = MultiScaleDiscriminator().to(DEVICE) 

criterion_GAN = nn.MSELoss()
criterion_cycle = nn.L1Loss()
criterion_identity = nn.L1Loss()

def gan_loss(preds, target):
    loss = 0
    for p in preds:
        loss += criterion_GAN(p, target)
    return loss

def feature_matching_loss_multi(fake, real, D, lambda_var=0.5):
    _, fake_feats_all = D(fake, return_features=True)
    _, real_feats_all = D(real, return_features=True)

    loss = 0.0

    for fake_feats, real_feats in zip(fake_feats_all, real_feats_all):
        for f_fake, f_real in zip(fake_feats, real_feats):
            f_real = f_real.detach()

            B, C, H, W = f_fake.shape
            f_fake = f_fake.view(B, C, -1)
            f_real = f_real.view(B, C, -1)

            mean_fake = f_fake.mean(dim=(0,2))
            mean_real = f_real.mean(dim=(0,2))

            var_fake = f_fake.var(dim=(0,2), unbiased=False)
            var_real = f_real.var(dim=(0,2), unbiased=False)

            loss += (mean_fake - mean_real).abs().mean()
            loss += lambda_var * (var_fake - var_real).abs().mean()

    return loss

opt_G = torch.optim.Adam(
    list(anime_G.parameters()) + list(real_G.parameters()),
    lr=0.0002, betas=(0.5, 0.999)
)

opt_D_A = torch.optim.Adam(anime_D.parameters(), lr=0.0003, betas=(0.5, 0.999))
opt_D_B = torch.optim.Adam(real_D.parameters(), lr=0.0003, betas=(0.5, 0.999))

decay_start_epoch = 120

def lambda_rule(epoch):
    return 1.0 - max(0, epoch - decay_start_epoch) / (EPOCH - decay_start_epoch)

lr_scheduler_G = torch.optim.lr_scheduler.LambdaLR(opt_G, lr_lambda=lambda_rule)
lr_scheduler_anime_D = torch.optim.lr_scheduler.LambdaLR(opt_D_A, lr_lambda=lambda_rule)
lr_scheduler_real_D  = torch.optim.lr_scheduler.LambdaLR(opt_D_B, lr_lambda=lambda_rule)

fake_anime_buffer = ReplayBuffer()
fake_real_buffer  = ReplayBuffer()

print(DEVICE)

In [ ]:

chk_pt_path = "/common/home/users/a/annamalaik.2022/cs-424-group-project-friday/work_dirs/Cycle_GAN_02/checkpoints/ckpt_74.pth"
if os.path.exists(chk_pt_path):
    ckpt = torch.load("/common/home/users/a/annamalaik.2022/cs-424-group-project-friday/work_dirs/Cycle_GAN_02/checkpoints/ckpt_74.pth", map_location=DEVICE)

    anime_G.load_state_dict(ckpt['anime_G'])
    real_G.load_state_dict(ckpt['real_G'])
    anime_D.load_state_dict(ckpt['anime_D'])
    real_D.load_state_dict(ckpt['real_D'])

    opt_G.load_state_dict(ckpt['optimizer_G'])
    opt_D_A.load_state_dict(ckpt['optimizer_anime_D'])
    opt_D_B.load_state_dict(ckpt['optimizer_real_D'])

    lr_scheduler_G.load_state_dict(ckpt['lr_scheduler_G'])
    lr_scheduler_anime_D.load_state_dict(ckpt['lr_scheduler_anime_D'])
    lr_scheduler_real_D.load_state_dict(ckpt['lr_scheduler_real_D'])

    start_epoch = ckpt['epoch'] + 1
else:
    start_epoch = 0
print(start_epoch)

In [ ]:
# -----------------------------
# TRAIN LOOP
# -----------------------------

print("Starting training...")

for epoch in range(start_epoch, EPOCH):
    for real_anime, real_face in dataloader:
        real_anime = real_anime.to(DEVICE)
        real_face = real_face.to(DEVICE)

        # ------------------ GENERATOR ------------------
        opt_G.zero_grad()
        
        # anime_G maps anime → real, 
        # real_G maps real → anime
        fake_face = anime_G(real_anime)
        fake_anime = real_G(real_face)

        preds_real = real_D(fake_face)
        loss_GAN_A = sum(criterion_GAN(p, torch.ones_like(p)) for p in preds_real)

        preds_anime = anime_D(fake_anime)
        loss_GAN_B = sum(criterion_GAN(p, torch.ones_like(p)) for p in preds_anime)

        rec_anime = real_G(fake_face)
        rec_face = anime_G(fake_anime)

        loss_cycle = criterion_cycle(rec_anime, real_anime) + \
                     criterion_cycle(rec_face, real_face)

        loss_id = criterion_identity(real_G(real_anime), real_anime) + \
                  criterion_identity(anime_G(real_face), real_face)

        loss_feat = feature_matching_loss_multi(fake_face, real_face, real_D) + \
                    feature_matching_loss_multi(fake_anime, real_anime, anime_D)

        loss_G = (lambda_gan * (loss_GAN_A + loss_GAN_B)
                  + lambda_cycle * loss_cycle
                  + lambda_id * loss_id
                  + lambda_feat * loss_feat
                  )

        loss_G.backward()
        opt_G.step()

        # -------- Discriminator anime --------
        opt_D_A.zero_grad()

        pred_real = anime_D(real_anime)
        loss_real = sum(criterion_GAN(p, torch.ones_like(p)) for p in pred_real)

        fake_buf = fake_anime_buffer.push_and_pop(fake_anime.detach())
        pred_fake = anime_D(fake_buf)
        loss_fake = sum(criterion_GAN(p, torch.zeros_like(p)) for p in pred_fake)

        loss_D_A = (loss_real + loss_fake) * 0.5
        loss_D_A.backward()
        opt_D_A.step()

        # -------- Discriminator real --------
        opt_D_B.zero_grad()

        pred_real = real_D(real_face)  # list of outputs
        loss_real = sum(
            criterion_GAN(p, torch.ones_like(p)) for p in pred_real
        )

        fake_buf = fake_real_buffer.push_and_pop(fake_face.detach())
        pred_fake = real_D(fake_buf)  # list of outputs
        loss_fake = sum(
            criterion_GAN(p, torch.zeros_like(p)) for p in pred_fake
        )

        loss_D_B = (loss_real + loss_fake) * 0.5
        loss_D_B.backward()
        opt_D_B.step()

    print(f"Epoch [{epoch+1}/{EPOCH}] | G: {loss_G.item():.4f}")

    lr_scheduler_G.step()
    lr_scheduler_anime_D.step()
    lr_scheduler_real_D.step()

    # -------------------------
    # Optional: Save sample images every 10 epochs
    # -------------------------
    if epoch % 10 == 0:
        with torch.no_grad():
            sample_anime = real_anime[:2]
            sample_face = real_face[:2]

            # Forward translations
            fake_face = anime_G(sample_anime)
            fake_anime = real_G(sample_face)

            # Cycle reconstruction
            recon_anime = real_G(fake_face)
            recon_face = anime_G(fake_anime)

            from torchvision.utils import save_image
            import torch

            # Helper to denormalize
            def denorm(x):
                return x * 0.5 + 0.5

            # ===== ANIME DOMAIN VIS =====
            anime_grid = torch.cat([
                denorm(sample_anime),   # real anime
                denorm(fake_face),      # fake face
                denorm(recon_anime)     # reconstructed anime
            ], dim=0)

            save_image(anime_grid, f"{OUTPUT_PATH}/epoch_{epoch}_anime_triplet.png", nrow=2)

            # ===== FACE DOMAIN VIS =====
            face_grid = torch.cat([
                denorm(sample_face),    # real face
                denorm(fake_anime),    # fake anime
                denorm(recon_face)     # reconstructed face
            ], dim=0)

            save_image(face_grid, f"{OUTPUT_PATH}/epoch_{epoch}_face_triplet.png", nrow=2)

    if (epoch + 1) % CHECKPOINT_SAVE_EVERY == 0:
        torch.save({
            'anime_G': anime_G.state_dict(),
            'real_G': real_G.state_dict(),
            'anime_D': anime_D.state_dict(),
            'real_D': real_D.state_dict(),
            'optimizer_G': opt_G.state_dict(),
            'optimizer_anime_D': opt_D_A.state_dict(),
            'optimizer_real_D': opt_D_B.state_dict(),
            'lr_scheduler_G': lr_scheduler_G.state_dict(),
            'lr_scheduler_anime_D': lr_scheduler_anime_D.state_dict(),
            'lr_scheduler_real_D': lr_scheduler_real_D.state_dict(),
            'epoch': epoch
        }, os.path.join(OUTPUT_PATH, 'checkpoints', f'ckpt_{epoch+1}.pth'))

In [ ]:
# ============================================================
# LOAD CHECKPOINT
# ============================================================

PATH_TO_CHECKPOINT = os.path.join(OUTPUT_PATH, "checkpoints", "ckpt_200.pth")

checkpoint = torch.load(PATH_TO_CHECKPOINT, map_location=DEVICE)

anime_G.load_state_dict(checkpoint['anime_G'])
real_G.load_state_dict(checkpoint['real_G'])

anime_G.eval()
real_G.eval()

# Mapping
G_A2B = anime_G   # anime → real
G_B2A = real_G    # real → anime

Tensor = torch.cuda.FloatTensor if DEVICE == "cuda" else torch.FloatTensor

# ============================================================
# Translation 1: Anime → Real
# ============================================================

test_dir = os.path.join(DATASET_PATH, 'anime', 'test')
files = [os.path.join(test_dir, f) for f in os.listdir(test_dir)]

save_dir = os.path.join(OUTPUT_PATH, "generated_real")
os.makedirs(save_dir, exist_ok=True)

to_image = transforms.ToPILImage()

for i in range(0, len(files), BATCH_SIZE):

    imgs = []
    for j in range(i, min(len(files), i + BATCH_SIZE)):
        img = Image.open(files[j]).convert("RGB")
        img = transform(img)
        imgs.append(img)

    imgs = torch.stack(imgs).type(Tensor)

    fake_imgs = G_A2B(imgs).detach().cpu()

    for j in range(fake_imgs.size(0)):
        img = fake_imgs[j]
        img = (img * 0.5 + 0.5).clamp(0, 1)

        _, name = os.path.split(files[i+j])
        torchvision.utils.save_image(img, os.path.join(save_dir, name))

gt_dir = os.path.join(DATASET_PATH, 'real', 'test')

metrics = torch_fidelity.calculate_metrics(
    input1=save_dir,
    input2=gt_dir,
    cuda=True,
    fid=True,
    isc=True
)

fid_1 = metrics["frechet_inception_distance"]
is_1 = metrics["inception_score_mean"]

print("Anime → Real")
print("FID:", fid_1)
print("IS:", is_1)

gms_1 = np.sqrt(fid_1 / is_1) if is_1 > 0 else 0
print("GMS:", gms_1)

# ============================================================
# Translation 2: Real → Anime
# ============================================================

test_dir = os.path.join(DATASET_PATH, 'real', 'test')
files = [os.path.join(test_dir, f) for f in os.listdir(test_dir)]

save_dir = os.path.join(OUTPUT_PATH, "generated_anime")
os.makedirs(save_dir, exist_ok=True)

for i in range(0, len(files), BATCH_SIZE):

    imgs = []
    for j in range(i, min(len(files), i + BATCH_SIZE)):
        img = Image.open(files[j]).convert("RGB")
        img = transform(img)
        imgs.append(img)

    imgs = torch.stack(imgs).type(Tensor)

    fake_imgs = G_B2A(imgs).detach().cpu()

    for j in range(fake_imgs.size(0)):
        img = fake_imgs[j]
        img = (img * 0.5 + 0.5).clamp(0, 1)

        _, name = os.path.split(files[i+j])
        torchvision.utils.save_image(img, os.path.join(save_dir, name))
    
    gt_dir = os.path.join(DATASET_PATH, 'anime', 'test')

metrics = torch_fidelity.calculate_metrics(
    input1=save_dir,
    input2=gt_dir,
    cuda=True,
    fid=True,
    isc=True
)

fid_2 = metrics["frechet_inception_distance"]
is_2 = metrics["inception_score_mean"]

print("\nReal → Anime")
print("FID:", fid_2)
print("IS:", is_2)

gms_2 = np.sqrt(fid_2 / is_2) if is_2 > 0 else 0
print("GMS:", gms_2)

# ============================================================
# FINAL SCORE
# ============================================================

final_gms = np.round((gms_1 + gms_2) / 2, 5)

print("\nFinal GMS:", final_gms)

df = pd.DataFrame({'id': [1], 'label': [float(final_gms)]})

csv_path = os.path.join(OUTPUT_PATH, "task2_submission.csv")
df.to_csv(csv_path, index=False)

print(f"CSV saved to {csv_path}")